In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.appName("Office_Products_Cleaning").getOrCreate()

your 131072x1 screen size is bogus. expect trouble
26/03/28 21:37:11 WARN Utils: Your hostname, kien resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/28 21:37:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 21:37:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
df = spark.read.format("json").option("inferSchema","true").load("./data/raw/Office_Products.jsonl")

In [4]:
initial_count = df.count()

In [5]:
df.show(5)


+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|      asin|helpful_vote|              images|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|
+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|B01AHHL4X2|           0|                  []| B01MZ3SD2X|   5.0|Lovely ink. Write...|1677939345945| Pretty & I love it!|AFKZENTNBQ7A7V7UX...|             true|
|B08L6H23JZ|           0|                  []| B08L6H23JZ|   4.0|Overall I’m prett...|1677939160682|2 excellent 1 ext...|AFKZENTNBQ7A7V7UX...|             true|
|B07JDZ5J46|           2|                  []| B07JDZ5J46|   1.0|[[VIDEOID:63276c1...|1660188831933|I don’t get the r...|AFKZENTNBQ7A7V7UX...|             true|
|B004MNX7EW|           0|[{IMAGE, 

In [6]:
df = df.withColumn("review_text", concat_ws(" ", col("title"),col("text")))

In [7]:
df = df.filter(col("review_text").isNotNull())
df = df.filter(trim(col("review_text")) != "")

In [8]:
df = df.dropDuplicates(["review_text","user_id","parent_asin"])

In [9]:
after_clean_count = df.count()

26/03/28 21:38:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


In [10]:
df = df.withColumn("review_text", regexp_replace(col("review_text"), "[^a-z0-9\\.\\?\\!\\s]", ""))
df = df.withColumn("review_text", regexp_replace(col("review_text"),"\\s+"," "))
df = df.withColumn("review_text", trim(col("review_text")))

In [11]:
df = df.withColumn("sentence_text", explode(split(col("review_text"), "[\\.\\!\\?]+")))
df = df.filter(trim(col("sentence_text")) != "")

In [12]:
df = df.withColumn("review_id", monotonically_increasing_id())

In [13]:
df = df.withColumn("sentence_id", row_number().over(Window.partitionBy("review_id").orderBy("sentence_text")))

In [14]:
df_final = df.select(
    "parent_asin",
    "review_id",
    "sentence_id",
    "sentence_text",
    "rating"
)

In [15]:
sentence_count = df_final.count()

26/03/28 21:39:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


In [ ]:
output_path = "./data/processed/Office_Products_Cleaned.parquet"

In [17]:
df_final.write.mode("overwrite").parquet(output_path)


26/03/28 21:41:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/28 21:41:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/28 21:41:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/28 21:41:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/28 21:41:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/28 21:41:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/28 21:41:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/28 21:41:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/28 21:41:36 WARN RowBasedKeyValueBatch: Calling spill() on

In [19]:
import os

In [42]:
def get_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        print(f"Directory: {dirpath}, Subdirectories: {dirnames}, Files: {filenames}")
        for f in filenames:
           
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

json_size = get_size("data/raw")
parquet_size = get_size("data/processed")

Directory: data/raw, Subdirectories: [], Files: ['Office_Products.jsonl']
Directory: data/processed, Subdirectories: ['Office_Products_Cleaned.parquet'], Files: []
Directory: data/processed/Office_Products_Cleaned.parquet, Subdirectories: [], Files: ['.part-00000-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.parquet.crc', '.part-00001-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.parquet.crc', '.part-00002-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.parquet.crc', '.part-00003-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.parquet.crc', '.part-00004-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.parquet.crc', '.part-00005-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.parquet.crc', '.part-00006-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.parquet.crc', '.part-00007-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.parquet.crc', '.part-00008-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.parquet.crc', '.part-00009-19eb84cc-48d3-4de7-8014-991a61242715-c000.snappy.par

In [44]:
report_path = "outputs/reports/reviews_cleaning1_report_office_products.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"Số review ban đầu: {initial_count}\n")
    f.write(f"Số review còn lại: {after_clean_count}\n")
    f.write(f"Số câu sau khi tách: {sentence_count}\n")
    f.write(f"Kích thước ban đầu: {json_size / (1024*1024):.2f} MB\n")
    f.write(f"Kích thước cuối cùng: {parquet_size / (1024*1024):.2f} MB\n")